In [2]:
import jax

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_platform_name", "gpu")
print(jax.local_devices()[0].device_kind)

import os
import zodiax as zdx
from zodiax.optimisation import sgd, adam
from jax import numpy as np, random as jr, tree as jtu
from scipy.ndimage import binary_dilation

# import dLux.utils as dlu

import amigo
import dorito

# import astropy

# visualisation
import matplotlib.pyplot as plt
import matplotlib as mpl


import ehtplot
import scienceplots
import cmasher as cmr

# matplotlib parameters
plt.style.use(["science", "bright", "no-latex"])

plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 8
plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

inferno = mpl.colormaps["inferno"]
viridis = mpl.colormaps["viridis"]
seismic = mpl.colormaps["seismic"]

inferno.set_bad("k", 0.5)
viridis.set_bad("k", 0.5)
seismic.set_bad("k", 0.5)

NVIDIA GeForce RTX 2080 Ti


## Load our model and fitted visibilities

In [4]:
from socket import gethostname

if gethostname() == "glinton":
    morgana = "/media/morgana1/"
else:
    morgana = "/Volumes/morgana1/"

data_path = os.path.join(morgana, "snert/max/data/JWST/WR137/calslope/")
uncal_path = os.path.join(morgana, "snert/max/data/JWST/WR137/uncal/")
amigo_cache = os.path.join(morgana, "snert/max/data/amigo_files/")

cache = os.path.join(amigo_cache, "cal_files/")
output_path = os.path.join(amigo_cache, "outputs/WR137/")

# Load the cached states
load_dict = lambda x: np.load(x, allow_pickle=True).item()
cal_values = load_dict(cache + "cal_model.npy")
vis_basis = load_dict(cache + "vis_basis.npy")
fit = load_dict(output_path + f"visibilities.npy")
n_basis = fit.pop("n_basis")

optics = amigo.optical_models.AMIOptics(psf_upsample=1)
vis_model = amigo.vis_models.LogVisModel(vis_basis, n_basis=n_basis)

2025-05-26 09:46:15.665729: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 336.00MiB (rounded to 352321536)requested by op 
2025-05-26 09:46:15.665895: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ****************************________**************************************************************xx
E0526 09:46:15.665905 1302904 pjrt_stream_executor_client.cc:2839] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 352321536 bytes. [tf-allocator-allocation-error='']


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 352321536 bytes.

# Get the average aberrations per filter for the Kernel visibilities